In [1]:
import os
import random
import numpy as np
import tensorflow as tf

# =========================
# 0) Reproducibility Settings
# =========================
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# For dataset
import tensorflow_datasets as tfds

# For models
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.layers import Conv2D, Dense, Flatten, Reshape, LeakyReLU, Dropout, Embedding, Input, Concatenate, Conv2DTranspose

# For callback
from tensorflow.keras.preprocessing.image import array_to_img
from tensorflow.keras.callbacks import Callback

# For streamlining ML workflow
import wandb
import math
import matplotlib.pyplot as plt

# Requires TensorFlow >=2.11 for the GroupNormalization layer.
from tensorflow import keras
from tensorflow.keras import layers

# For model memory usage
from tensorflow.keras import backend as K
import humanize

from tensorflow.keras.layers import (Input, Dense, Reshape, Embedding, Flatten,
                                     Concatenate, UpSampling2D, Conv2D, BatchNormalization,
                                     LeakyReLU, Add)
import time
import glob
from PIL import Image

num_epochs = 100
total_timesteps = 1000
norm_groups = 8 
learning_rate = 1e-4
batch_size = 16
img_size = 128
img_channels = 3

# =========================
# Architecture Settings
# =========================
first_conv_channels = 64
channel_multiplier = [1, 2, 4, 8]
widths = [first_conv_channels * m for m in channel_multiplier]
has_attention = [False, False, True, True]
num_res_blocks = 2

# =========================
# Diffusion Logic
# =========================
class GaussianDiffusion:
    def __init__(
        self,
        timesteps=1000,
        clip_min=-1.0,
        clip_max=1.0,
    ):
        self.timesteps = timesteps
        self.clip_min = clip_min
        self.clip_max = clip_max

        # 1. Cosine Schedule Function
        def f(t, T):
            s = 0.008
            return np.cos((((t / T) + s) / (1 + s)) * (np.pi / 2)) ** 2

        # 2. Calculate alphas_cumprod
        t = np.arange(timesteps + 1, dtype=np.float64)
        alphas_cumprod = f(t, timesteps) / f(0, timesteps)
        
        # 3. Derive betas and clip to prevent singularities
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        betas = np.clip(betas, a_min=0, a_max=0.999)
        self.betas = tf.constant(betas, dtype=tf.float32)
        
        alphas = 1.0 - betas
        alphas_cumprod = np.cumprod(alphas, axis=0)
        alphas_cumprod_prev = np.append(1.0, alphas_cumprod[:-1])

        self.num_timesteps = int(timesteps)
        self.alphas_cumprod = tf.constant(alphas_cumprod, dtype=tf.float32)
        self.alphas_cumprod_prev = tf.constant(alphas_cumprod_prev, dtype=tf.float32)

        # Calculations for diffusion q(x_t | x_{t-1})
        self.sqrt_alphas_cumprod = tf.constant(np.sqrt(alphas_cumprod), dtype=tf.float32)
        self.sqrt_one_minus_alphas_cumprod = tf.constant(np.sqrt(1.0 - alphas_cumprod), dtype=tf.float32)
        self.log_one_minus_alphas_cumprod = tf.constant(np.log(1.0 - alphas_cumprod), dtype=tf.float32)
        self.sqrt_recip_alphas_cumprod = tf.constant(np.sqrt(1.0 / alphas_cumprod), dtype=tf.float32)
        self.sqrt_recipm1_alphas_cumprod = tf.constant(np.sqrt(1.0 / alphas_cumprod - 1), dtype=tf.float32)

        # Calculations for posterior q(x_{t-1} | x_t, x_0)
        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        self.posterior_variance = tf.constant(posterior_variance, dtype=tf.float32)

        self.posterior_log_variance_clipped = tf.constant(
            np.log(np.maximum(posterior_variance, 1e-20)), dtype=tf.float32
        )

        self.posterior_mean_coef1 = tf.constant(
            betas * np.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod), dtype=tf.float32,
        )

        self.posterior_mean_coef2 = tf.constant(
            (1.0 - alphas_cumprod_prev) * np.sqrt(alphas) / (1.0 - alphas_cumprod), dtype=tf.float32,
        )
        
        self.snr = self.alphas_cumprod / (1.0 - self.alphas_cumprod)

    def _extract(self, a, t, x_shape):
        batch_size = x_shape[0]
        out = tf.gather(a, t)
        return tf.reshape(out, [batch_size, 1, 1, 1])

    def q_mean_variance(self, x_start, t):
        x_start_shape = tf.shape(x_start)
        mean = self._extract(self.sqrt_alphas_cumprod, t, x_start_shape) * x_start
        variance = self._extract(1.0 - self.alphas_cumprod, t, x_start_shape)
        log_variance = self._extract(self.log_one_minus_alphas_cumprod, t, x_start_shape)
        return mean, variance, log_variance

    def q_sample(self, x_start, t, noise):
        x_start_shape = tf.shape(x_start)
        return (
            self._extract(self.sqrt_alphas_cumprod, t, tf.shape(x_start)) * x_start
            + self._extract(self.sqrt_one_minus_alphas_cumprod, t, x_start_shape) * noise
        )

    def predict_start_from_noise(self, x_t, t, noise):
        x_t_shape = tf.shape(x_t)
        return (
            self._extract(self.sqrt_recip_alphas_cumprod, t, x_t_shape) * x_t
            - self._extract(self.sqrt_recipm1_alphas_cumprod, t, x_t_shape) * noise
        )

    def q_posterior(self, x_start, x_t, t):
        x_t_shape = tf.shape(x_t)
        posterior_mean = (
            self._extract(self.posterior_mean_coef1, t, x_t_shape) * x_start
            + self._extract(self.posterior_mean_coef2, t, x_t_shape) * x_t
        )
        posterior_variance = self._extract(self.posterior_variance, t, x_t_shape)
        posterior_log_variance_clipped = self._extract(
            self.posterior_log_variance_clipped, t, x_t_shape
        )
        return posterior_mean, posterior_variance, posterior_log_variance_clipped

    def p_mean_variance(self, pred_noise, x, t, clip_denoised=True):
        x_recon = self.predict_start_from_noise(x, t=t, noise=pred_noise)
        
        if clip_denoised:
            x_recon = self.dynamic_thresholding(x_recon, p=0.995, c=1.0)

        model_mean, posterior_variance, posterior_log_variance = self.q_posterior(
            x_start=x_recon, x_t=x, t=t
        )
        return model_mean, posterior_variance, posterior_log_variance

    def p_sample(self, pred_noise, x, t, clip_denoised=True):
        model_mean, _, model_log_variance = self.p_mean_variance(
            pred_noise, x=x, t=t, clip_denoised=clip_denoised
        )
        noise = tf.random.normal(shape=x.shape, dtype=x.dtype)
        nonzero_mask = tf.reshape(
            1 - tf.cast(tf.equal(t, 0), tf.float32), [tf.shape(x)[0], 1, 1, 1]
        )
        return model_mean + nonzero_mask * tf.exp(0.5 * model_log_variance) * noise
        
    def dynamic_thresholding(self, x_recon, p=0.995, c=1.0):
        abs_x = tf.abs(x_recon)
        batch_size = tf.shape(abs_x)[0]
        abs_x_flat = tf.reshape(abs_x, [batch_size, -1])
        
        sorted_x = tf.sort(abs_x_flat, axis=-1)
        num_elements = tf.cast(tf.shape(sorted_x)[-1], tf.float32)
        
        percentile_index = tf.cast(num_elements * p, tf.int32)
        percentile_index = tf.clip_by_value(percentile_index, 0, tf.shape(sorted_x)[-1] - 1) 
        
        s = tf.gather(sorted_x, percentile_index, axis=-1)
        s = tf.maximum(s, c)
        s = tf.reshape(s, [batch_size, 1, 1, 1])
        
        x_recon = tf.clip_by_value(x_recon, -s, s)
        x_recon = x_recon / s
        return x_recon

# =========================
# Model Building Functions
# =========================
def kernel_init(scale):
    scale = max(scale, 1e-10)
    return keras.initializers.VarianceScaling(
        scale, mode="fan_avg", distribution="uniform"
    )

class AttentionBlock(layers.Layer):
    def __init__(self, units, groups=8, **kwargs):
        self.units = units
        self.groups = groups
        super().__init__(**kwargs)
        self.norm = layers.GroupNormalization(groups=groups)
        self.query = layers.Dense(units, kernel_initializer=kernel_init(1.0))
        self.key = layers.Dense(units, kernel_initializer=kernel_init(1.0))
        self.value = layers.Dense(units, kernel_initializer=kernel_init(1.0))
        self.proj = layers.Dense(units, kernel_initializer=kernel_init(0.0))

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        height = tf.shape(inputs)[1]
        width = tf.shape(inputs)[2]
        scale = tf.cast(self.units, tf.float32) ** (-0.5)

        inputs = self.norm(inputs)
        q = self.query(inputs)
        k = self.key(inputs)
        v = self.value(inputs)

        attn_score = tf.einsum("bhwc, bHWc->bhwHW", q, k) * scale
        attn_score = tf.reshape(attn_score, [batch_size, height, width, height * width])
        attn_score = tf.nn.softmax(attn_score, -1)
        attn_score = tf.reshape(attn_score, [batch_size, height, width, height, width])

        proj = tf.einsum("bhwHW,bHWc->bhwc", attn_score, v)
        proj = self.proj(proj)
        return inputs + proj

class TimeEmbedding(layers.Layer):
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.half_dim = dim // 2
        self.emb = math.log(10000) / (self.half_dim - 1)
        self.emb = tf.exp(tf.range(self.half_dim, dtype=tf.float32) * -self.emb)

    def call(self, inputs):
        inputs = tf.cast(inputs, dtype=tf.float32)
        emb = inputs[:, None] * self.emb[None, :]
        emb = tf.concat([tf.sin(emb), tf.cos(emb)], axis=-1)
        return emb

def ResidualBlock(width, groups=8, activation_fn=keras.activations.swish):
    def apply(inputs):
        x, t = inputs
        input_width = x.shape[3]
        if input_width == width:
            residual = x
        else:
            residual = layers.Conv2D(width, kernel_size=1, kernel_initializer=kernel_init(1.0))(x)

        temb = activation_fn(t)
        temb = layers.Dense(width, kernel_initializer=kernel_init(1.0))(temb)[:, None, None, :]

        x = layers.GroupNormalization(groups=groups)(x)
        x = activation_fn(x)
        x = layers.Conv2D(width, kernel_size=3, padding="same", kernel_initializer=kernel_init(1.0))(x)

        x = layers.Add()([x, temb])
        x = layers.GroupNormalization(groups=groups)(x)
        x = activation_fn(x)

        x = layers.Conv2D(width, kernel_size=3, padding="same", kernel_initializer=kernel_init(0.0))(x)
        x = layers.Add()([x, residual])
        return x
    return apply

def DownSample(width):
    def apply(x):
        x = layers.Conv2D(width, kernel_size=3, strides=2, padding="same", kernel_initializer=kernel_init(1.0))(x)
        return x
    return apply

def UpSample(width, interpolation="nearest"):
    def apply(x):
        x = layers.UpSampling2D(size=2, interpolation=interpolation)(x)
        x = layers.Conv2D(width, kernel_size=3, padding="same", kernel_initializer=kernel_init(1.0))(x)
        return x
    return apply

def TimeMLP(units, activation_fn=keras.activations.swish):
    def apply(inputs):
        temb = layers.Dense(units, activation=activation_fn, kernel_initializer=kernel_init(1.0))(inputs)
        temb = layers.Dense(units, kernel_initializer=kernel_init(1.0))(temb)
        return temb
    return apply

def build_model(
    img_size, img_channels, widths, has_attention, first_conv_channels,
    num_res_blocks, norm_groups=8, interpolation="nearest", activation_fn=keras.activations.swish,
):
    image_input = layers.Input(shape=(img_size, img_size, img_channels), name="image_input")
    time_input = keras.Input(shape=(), dtype=tf.int64, name="time_input")

    x = layers.Conv2D(first_conv_channels, kernel_size=(3, 3), padding="same", kernel_initializer=kernel_init(1.0))(image_input)

    temb = TimeEmbedding(dim=first_conv_channels * 4)(time_input)
    temb = TimeMLP(units=first_conv_channels * 4, activation_fn=activation_fn)(temb)

    skips = [x]

    # DownBlock
    for i in range(len(widths)):
        for _ in range(num_res_blocks):
            x = ResidualBlock(widths[i], groups=norm_groups, activation_fn=activation_fn)([x, temb])
            if has_attention[i]:
                x = AttentionBlock(widths[i], groups=norm_groups)(x)
            skips.append(x)
        if widths[i] != widths[-1]:
            x = DownSample(widths[i])(x)
            skips.append(x)

    # MiddleBlock
    x = ResidualBlock(widths[-1], groups=norm_groups, activation_fn=activation_fn)([x, temb])
    x = AttentionBlock(widths[-1], groups=norm_groups)(x)
    x = ResidualBlock(widths[-1], groups=norm_groups, activation_fn=activation_fn)([x, temb])

    # UpBlock
    for i in reversed(range(len(widths))):
        for _ in range(num_res_blocks + 1):
            x = layers.Concatenate(axis=-1)([x, skips.pop()])
            x = ResidualBlock(widths[i], groups=norm_groups, activation_fn=activation_fn)([x, temb])
            if has_attention[i]:
                x = AttentionBlock(widths[i], groups=norm_groups)(x)
        if i != 0:
            x = UpSample(widths[i], interpolation=interpolation)(x)

    # End block
    x = layers.GroupNormalization(groups=norm_groups)(x)
    x = activation_fn(x)
    x = layers.Conv2D(3, (3, 3), padding="same", kernel_initializer=kernel_init(0.0))(x)
    return keras.Model([image_input, time_input], x, name="unet")

class DiffusionModel(keras.Model):
    def __init__(self, network, ema_network, timesteps ,img_size, img_channels, gdf_util, ema=0.999):
        super().__init__()
        self.network = network
        self.ema_network = ema_network
        self.timesteps = timesteps
        self.img_size = img_size        
        self.img_channels = img_channels
        self.gdf_util = gdf_util
        self.ema = ema

    def train_step(self, images):
        images = images[0] if isinstance(images, (tuple, list)) else images
        batch_size = tf.shape(images)[0]

        t = tf.random.uniform(minval=0, maxval=self.timesteps, shape=(batch_size,), dtype=tf.int64)

        with tf.GradientTape() as tape:
            # ---> INTEGRATED: OFFSET NOISE <---
            noise = tf.random.normal(shape=tf.shape(images), dtype=images.dtype)
            offset_strength = 0.1
            offset_noise = tf.random.normal(shape=(batch_size, 1, 1, self.img_channels), dtype=images.dtype)
            noise = noise + offset_strength * offset_noise
            
            images_t = self.gdf_util.q_sample(images, t, noise)
            pred_noise = self.network([images_t, t], training=True)
            
            # ---> INTEGRATED: MIN-SNR LOSS <---
            snr_t = self.gdf_util._extract(self.gdf_util.snr, t, tf.shape(images))
            gamma = 5.0
            loss_weight = tf.minimum(gamma, snr_t) / snr_t
            
            mse = tf.square(noise - pred_noise)
            loss = tf.reduce_mean(loss_weight * mse)

        gradients = tape.gradient(loss, self.network.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.network.trainable_weights))

        for weight, ema_weight in zip(self.network.weights, self.ema_network.weights):
            ema_weight.assign(self.ema * ema_weight + (1 - self.ema) * weight)

        return {"loss": loss}
        
    def generate_images(self, num_images=16):
        samples = tf.random.normal(
            shape=(num_images, self.img_size, self.img_size, self.img_channels), dtype=tf.float32
        )
        for t in reversed(range(0, self.timesteps)):
            tt = tf.cast(tf.fill(num_images, t), dtype=tf.int64)
            pred_noise = self.ema_network.predict([samples, tt], verbose=0, batch_size=num_images)
            samples = self.gdf_util.p_sample(pred_noise, samples, tt, clip_denoised=True)
        return samples
        
class ImageVisualizationCallback(keras.callbacks.Callback):
        def __init__(self, num_images=16, freq=10):
            super().__init__()
            self.num_images = num_images
            self.freq = freq

        def on_epoch_end(self, epoch, logs=None):
            if (epoch + 1) % self.freq == 0:
                print(f"\nGenerating images at the end of epoch {epoch + 1}...")
                
                generated_images = self.model.generate_images(num_images=self.num_images)
                
                generated_images = (generated_images + 1.0) / 2.0
                generated_images = generated_images.numpy()
                
                # ---> INTEGRATED: MATPLOTLIB CLIPPING FIX <---
                generated_images = np.clip(generated_images, 0.0, 1.0)
                
                num_cols = 4
                num_rows = self.num_images // num_cols
                plt.figure(figsize=(num_cols * 3, num_rows * 3))
                for i in range(self.num_images):
                    plt.subplot(num_rows, num_cols, i + 1)
                    plt.imshow(generated_images[i])
                    plt.axis("off")
                plt.suptitle(f"Generated Images at Epoch {epoch + 1}", fontsize=16)
                plt.tight_layout()
                plt.show()
                
class EMASaveCallback(keras.callbacks.Callback):
    def __init__(self, save_epochs, filepath_template="/kaggle/working/diffusion_model_epoch_{}.keras"):
        super().__init__()
        self.save_epochs = save_epochs
        self.filepath_template = filepath_template

    def on_epoch_end(self, epoch, logs=None):
        # Epochs are 0-indexed, so epoch 59 is the 60th epoch
        current_epoch = epoch + 1
        
        if current_epoch in self.save_epochs:
            # Dynamically format the filename to include the current epoch number
            current_filepath = self.filepath_template.format(current_epoch)
            
            print(f"\n[INFO] Reached Epoch {current_epoch}! Saving EMA Network to {current_filepath}...")
            # We save the ema_network specifically, as it is used for generation
            self.model.ema_network.save(current_filepath)

# =========================
# Execution
# =========================
network = build_model(
    img_size=img_size,
    img_channels=img_channels,
    first_conv_channels=first_conv_channels,
    widths=widths,
    has_attention=has_attention,
    num_res_blocks=num_res_blocks,
    norm_groups=norm_groups,
    activation_fn=keras.activations.swish,
)

ema_network = build_model(
    img_size=img_size,
    img_channels=img_channels,
    first_conv_channels=first_conv_channels,
    widths=widths,
    has_attention=has_attention,
    num_res_blocks=num_res_blocks,
    norm_groups=norm_groups,
    activation_fn=keras.activations.swish,
)
ema_network.set_weights(network.get_weights())

# Timesteps initialized for Cosine Schedule
gdf_util = GaussianDiffusion(timesteps=total_timesteps)

model = DiffusionModel(
    network=network,
    ema_network=ema_network,
    gdf_util=gdf_util,
    timesteps=total_timesteps,
    img_size=img_size,  
    img_channels=img_channels,
)

model.compile(
    loss=keras.losses.MeanSquaredError(),
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0),
)

viz_callback = ImageVisualizationCallback(num_images=16, freq=10)
ema_save_callback = EMASaveCallback(save_epochs=[60, 70, 80, 90, 100])

'''# Train the model with BOTH callbacks
start_time = time.time()
history = model.fit(
    dataset,
    epochs=num_epochs,
    steps_per_epoch=len(file_paths) // batch_size,
    callbacks=[ema_save_callback] 
)
end_time = time.time()

ema_network.save("/kaggle/working/diffusion_model_128.keras")

print("Model saved successfully")

training_time = end_time - start_time
print("Training time (seconds):", training_time)
print("Training time (minutes):", training_time/60)
print("Training time (hours):", training_time/3600)'''

2026-05-07 11:26:08.664979: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778153168.882125      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778153168.943691      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778153169.395040      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778153169.395077      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778153169.395080      57 computation_placer.cc:177] computation placer alr

'# Train the model with BOTH callbacks\nstart_time = time.time()\nhistory = model.fit(\n    dataset,\n    epochs=num_epochs,\n    steps_per_epoch=len(file_paths) // batch_size,\n    callbacks=[ema_save_callback] \n)\nend_time = time.time()\n\nema_network.save("/kaggle/working/diffusion_model_128.keras")\n\nprint("Model saved successfully")\n\ntraining_time = end_time - start_time\nprint("Training time (seconds):", training_time)\nprint("Training time (minutes):", training_time/60)\nprint("Training time (hours):", training_time/3600)'

In [ ]:

CLASS_NAME = "STR" 
NUM_IMAGES_TO_GENERATE = 3000 
BATCH_SIZE = 32
SAVE_DIR = f"/kaggle/working/synthetic_data/{CLASS_NAME}/"
MODEL_WEIGHTS_PATH = "/kaggle/input/models/snowbubble0/str-100/keras/default/1/diffusion_model_epoch_100_STR.keras"


os.makedirs(SAVE_DIR, exist_ok=True)


print(f"Loading diffusion model weights for {CLASS_NAME}...")

model.ema_network.load_weights(MODEL_WEIGHTS_PATH)

# 3. Generation Loop
print(f"Generating {NUM_IMAGES_TO_GENERATE} synthetic images...")
num_batches = int(np.ceil(NUM_IMAGES_TO_GENERATE / BATCH_SIZE))
images_saved = 0

for i in tqdm(range(num_batches), desc="Generating Batches"):
    # Calculate how many images to generate in this specific batch
    # (The last batch might be smaller than 32 to hit the exact target number)
    current_batch_size = min(BATCH_SIZE, NUM_IMAGES_TO_GENERATE - images_saved)
    
    # Generate the images
    fake_images = model.generate_images(num_images=current_batch_size)
    
    # Save each image to the folder
    for j in range(current_batch_size):
        # Scale the image from [-1, 1] back to [0, 255] for saving
        img_array = tf.keras.preprocessing.image.array_to_img(fake_images[j])
        
        # Create a unique filename
        filename = f"synthetic_{CLASS_NAME}_{images_saved + 1:05d}.png"
        filepath = os.path.join(SAVE_DIR, filename)
        
        # Save to disk
        img_array.save(filepath)
        images_saved += 1

print(f"\{images_saved} images saved to: {SAVE_DIR}")

Loading diffusion model weights for STR...
Generating 3000 synthetic images...


Generating Batches:   0%|          | 0/94 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1778153208.996610     128 service.cc:152] XLA service 0x7ef908003800 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778153208.996645     128 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1778153210.315924     128 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1778153233.491214     128 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
Generating Batches: 100%|██████████| 94/94 [9:46:18<00:00, 374.24s/it]  


✅ Success! 3000 images saved to: /kaggle/working/synthetic_data/STR/


In [3]:
import shutil
import os

# 1. Setup the paths
folder_to_zip = "/kaggle/working/synthetic_data/STR"
output_zip_name = "/kaggle/working/synthetic_STR_images"

# 2. Compress the folder
print(f"Compressing images from {folder_to_zip}...")
shutil.make_archive(output_zip_name, 'zip', folder_to_zip)

print(f"file is ready: {output_zip_name}.zip")

Compressing images from /kaggle/working/synthetic_data/STR...
✅ Success! Your file is ready: /kaggle/working/synthetic_STR_images.zip
